# RHI Live Runtime v24 — Inverse Retrieval H-Field Pressure Test

Δ **Purpose:** test whether inverse-retrieval branches show stronger continuous H-field pressure than null expectation.

v23 established the clean boundary:

$$
\boxed{
\text{H-crossing exists, but H-residence was not established.}
}
$$

The next move is not another best-hit scan. v24 tests continuous pressure:

$$
P_H(\sigma)=\frac{1}{L}\sum_{\ell=1}^{L}
\exp\left(-\frac{|x_\ell-H|}{\sigma}\right)
$$

Primary pre-registered channel:

$$
x_\ell = H_\ell^{(\text{entropy})}
$$

Primary profile:

$$
\text{inverse\_retrieval}
$$

H remains a readout/destination, not a steering target.

Outputs exactly two files:

1. `rhi_v24_<run_id>_bundle.json`
2. `rhi_v24_<run_id>_summary.csv`


In [1]:

from __future__ import annotations

import os, re, sys, json, math, uuid, time, random, traceback, subprocess, importlib
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v24_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v24_" + uuid.uuid4().hex[:10]
SEED = 24
random.seed(SEED)
np.random.seed(SEED)

# Model settings.
MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True
AUTO_INSTALL_MISSING_DEPS = True

# Runtime scale. Increase RUN_PROMPT_LIMIT after first successful run.
RUN_PROMPT_LIMIT = 36
BRANCH_ROLES = ["construct", "verify"]
MAX_NEW_TOKENS = 120
TEMPERATURE = 0.7

# H is a readout, not a steering target.
H_TARGET = math.pi / 9

# Primary pressure settings.
PRIMARY_PROFILE = "inverse_retrieval"
PRIMARY_CHANNEL = "entropy"
SIGMAS = [0.005, 0.01, 0.02, 0.05]

# Nulls.
N_TARGET_NULL = 1000     # random target values on same trajectory
N_MATCHED_NULL = 1000    # matched synthetic trajectories
N_POSITION_NULL = 1000   # order null for rolling-window pressure

# Rolling windows for local pressure concentration.
WINDOWS = [5, 10, 20]

print("RHI v24 — Inverse Retrieval H-Field Pressure")
print("RUN_ID:", RUN_ID)
print("H_TARGET:", H_TARGET)
print("PRIMARY:", PRIMARY_PROFILE, PRIMARY_CHANNEL)
print("OUT_DIR:", OUT_DIR)


RHI v24 — Inverse Retrieval H-Field Pressure
RUN_ID: rhi_v24_19f4edff63
H_TARGET: 0.3490658503988659
PRIMARY: inverse_retrieval entropy
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v24_outputs


## 1. Dependency and Model Load

The model is used only to generate token-level fold trajectories. The H value is never injected into the prompts.


In [2]:

def _module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False
    except Exception:
        return False

def ensure_runtime_dependencies() -> Dict[str, Any]:
    status = {
        "checked": True,
        "attempted_install": False,
        "missing_before": [],
        "missing_after": [],
        "errors": []
    }
    required = [
        ("torch", "torch"),
        ("transformers", "transformers"),
        ("sentencepiece", "sentencepiece"),
        ("google.protobuf", "protobuf"),
    ]
    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_before"].append(pip_name)

    if status["missing_before"] and AUTO_INSTALL_MISSING_DEPS:
        status["attempted_install"] = True
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", *sorted(set(status["missing_before"]))],
                check=True
            )
            importlib.invalidate_caches()
        except Exception as e:
            status["errors"].append(repr(e))

    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_after"].append(pip_name)

    return status

DEPENDENCY_STATUS = ensure_runtime_dependencies()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_device_info() -> Dict[str, Any]:
    info = {
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "cuda_version": getattr(torch.version, "cuda", None),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
    return info

DEVICE_INFO = get_device_info()

model = None
tokenizer = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
SMOKE_TEXT = None

def load_model():
    global model, tokenizer, MODEL_READY, MODEL_ERROR
    if not LOAD_REAL_MODEL:
        return
    try:
        print("Loading model:", MODEL_ID_OR_PATH)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID_OR_PATH,
            dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if not torch.cuda.is_available():
            model = model.to("cpu")
        model.eval()
        MODEL_READY = True
        print("MODEL_READY:", MODEL_READY, "DEVICE:", next(model.parameters()).device)
    except Exception as e:
        MODEL_ERROR = traceback.format_exc()
        MODEL_READY = False
        print("MODEL LOAD ERROR:", repr(e))

def model_smoke_test():
    global MODEL_GENERATION_READY, SMOKE_TEXT, MODEL_ERROR
    if not MODEL_READY:
        MODEL_GENERATION_READY = False
        return
    try:
        messages = [{"role": "user", "content": "Reply with READY only."}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                return_dict_in_generate=True,
            )
        gen = out.sequences[0][inputs.input_ids.shape[1]:]
        SMOKE_TEXT = tokenizer.decode(gen, skip_special_tokens=True).strip()
        MODEL_GENERATION_READY = bool(SMOKE_TEXT)
        print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
        print("SMOKE:", SMOKE_TEXT)
    except Exception:
        MODEL_ERROR = traceback.format_exc()
        MODEL_GENERATION_READY = False
        print("SMOKE TEST FAILED")
        print(MODEL_ERROR)

load_model()
model_smoke_test()

print("DEPENDENCY_STATUS:", DEPENDENCY_STATUS)
print("DEVICE_INFO:", DEVICE_INFO)


Loading model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True DEVICE: cuda:0
MODEL_GENERATION_READY: True
SMOKE: READY
DEPENDENCY_STATUS: {'checked': True, 'attempted_install': False, 'missing_before': [], 'missing_after': [], 'errors': []}
DEVICE_INFO: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'cuda_version': '12.6', 'gpu_name': 'NVIDIA GeForce RTX 4060'}


## 2. Prompt Battery

v24 focuses on inverse retrieval. These prompts force retrieval by missing operation, function, need-slot, inverse fit, or preserved action rather than label/noun overlap.


In [3]:

INVERSE_RETRIEVAL_PROMPTS = [
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "how should retrieval work when keywords fail but the operation is obvious",
    "explain inverse operational fit for search without noun matching",
    "design a verifier for retrieval candidates selected by need rather than label",
    "how can an agent rank candidates by function instead of name",
    "describe retrieval by missing slot rather than surface term",
    "build a retrieval step that rejects keyword-only matches",
    "explain why literal shape examples are wrong for shape-first retrieval",
    "design search that retrieves by function not title",
    "how should a retriever find the operation when the label is absent",
    "explain candidate generation from inverse need",
    "describe ranking by preserved function instead of noun overlap",
    "build a retrieval stage that starts from the action the user is trying to complete",
    "explain how to retrieve a document when the name is wrong but the task need is clear",
    "design an inverse-need retriever for missing API documentation",
    "how should an agent search when the user describes the failure but not the file name",
    "describe retrieval by constraint satisfaction instead of keyword similarity",
    "build a search verifier that rejects semantically adjacent but operationally wrong results",
    "explain how function-preserving retrieval differs from semantic similarity",
    "design a retrieval gate that checks whether a candidate can actually perform the requested operation",
    "how can a retriever find the hidden tool when the user only states the desired effect",
    "describe inverse search from output requirement to input artifact",
    "build a retrieval path for when the noun is absent but the verb is stable",
    "explain how to retrieve by transformation rather than object label",
    "design a search system that prioritizes operational affordance over title match",
    "how should an agent retrieve prior work when only the unresolved slot is known",
    "describe retrieval by causal role in a workflow",
    "build a verifier for candidates chosen because they close the missing step",
    "explain how to avoid false positives in shape-first retrieval",
    "design candidate generation from a negative space description",
    "how should retrieval work when the query describes what is missing",
    "describe ranking documents by what they enable the agent to do",
    "build an inverse retrieval plan for finding code that fixes an error without matching the error text",
    "explain why noun overlap can be a trap in agent retrieval",
    "design a fold-first retriever that preserves the operation across different labels",
    "how can retrieval identify the correct artifact from its downstream use",
    "describe a search process that starts from the intended state transition",
    "build a retrieval check that asks whether the result closes the loop",
    "explain retrieval as solving for the missing operand in an action chain",
    "design a retriever that treats the prompt as a constraint field, not a bag of words",
    "how should an agent locate evidence when the surface label is misleading",
    "describe a need-slot retrieval algorithm",
    "build a retrieval step that maps user intent into candidate affordances",
    "explain how inverse retrieval can recover context after vocabulary drift",
    "design a search loop that tests retrieved candidates against the requested operation",
    "how should a retriever handle synonyms that preserve nouns but break the action",
    "describe retrieval by conserved function across wording changes",
    "build a retrieval gate for selecting the result that repairs the user's workflow",
]

PROMPTS = INVERSE_RETRIEVAL_PROMPTS[:RUN_PROMPT_LIMIT]
print("Prompt count:", len(PROMPTS), "Branches:", BRANCH_ROLES, "Total generations:", len(PROMPTS) * len(BRANCH_ROLES))


Prompt count: 36 Branches: ['construct', 'verify'] Total generations: 72


## 3. Fold-Instrumented Generation

For each generated token, v24 stores only compact fold telemetry: entropy ratio, confidence, token, and pressure readouts. It does not store logits.


In [4]:

SCORER_TERMS = {
    "precondition", "rollback", "criteria", "boundary", "contract",
    "validation", "constraint", "invariant", "audit", "verify",
    "policy", "controller", "evidence"
}

def softmax_np(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    logits = logits / max(temperature, 1e-9)
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)

def entropy_and_confidence(logits: np.ndarray, temperature: float = 1.0) -> Tuple[float, float]:
    probs = softmax_np(logits, temperature)
    probs = np.clip(probs, 1e-12, 1.0)
    entropy = -float(np.sum(probs * np.log(probs)))
    confidence = float(np.max(probs))
    return entropy, confidence

def is_scorer_term(token_text: str) -> bool:
    t = token_text.lower().strip()
    return any(term in t for term in SCORER_TERMS)

def h_field_pressure(values: np.ndarray, target: float = H_TARGET, sigma: float = 0.01) -> np.ndarray:
    return np.exp(-np.abs(values - target) / max(sigma, 1e-12))

def rolling_mean(arr: np.ndarray, window: int) -> np.ndarray:
    if len(arr) < window or window <= 1:
        return arr.copy()
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode="valid")

def branch_prompt(base_prompt: str, role: str) -> str:
    if role == "construct":
        return (
            base_prompt
            + "\n\nAnswer directly. Emphasize the operation being preserved, not surface labels."
        )
    if role == "verify":
        return (
            base_prompt
            + "\n\nVerify the operational need first, reject noun-only matches, then answer."
        )
    return base_prompt

def generate_fold_log(prompt: str, branch_role: str, branch_id: str) -> Dict[str, Any]:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = out.sequences[0][inputs.input_ids.shape[1]:]
    scores = out.scores

    raw_entropies = []
    confidences = []
    fold_states = []

    # First pass raw entropy/confidence.
    for pos, (token_id, logits_tensor) in enumerate(zip(generated_ids, scores), start=1):
        logits = logits_tensor[0].detach().float().cpu().numpy()
        ent, conf = entropy_and_confidence(logits, TEMPERATURE)
        raw_entropies.append(ent)
        confidences.append(conf)

    max_entropy = max(raw_entropies) if raw_entropies else 1.0
    entropy_ratios = [ent / max_entropy if max_entropy > 0 else 0.0 for ent in raw_entropies]

    for pos, token_id in enumerate(generated_ids, start=1):
        token_text = tokenizer.decode([int(token_id.item())])
        entropy_ratio = float(entropy_ratios[pos - 1])
        confidence = float(confidences[pos - 1])

        state = {
            "position": pos,
            "token_id": int(token_id.item()),
            "token_text": token_text,
            "entropy_raw": float(raw_entropies[pos - 1]),
            "entropy_ratio": entropy_ratio,
            "confidence": confidence,
            "is_scorer_term": bool(is_scorer_term(token_text)),
            "entropy_h_distance": float(abs(entropy_ratio - H_TARGET)),
            "confidence_h_distance": float(abs(confidence - H_TARGET)),
        }
        fold_states.append(state)

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "branch_id": branch_id,
        "branch_role": branch_role,
        "prompt": prompt,
        "profile": PRIMARY_PROFILE,
        "total_tokens": len(fold_states),
        "final_text": gen_text,
        "fold_states": fold_states,
        "scorer_term_count": int(sum(s["is_scorer_term"] for s in fold_states)),
        "mean_entropy_ratio": float(np.mean(entropy_ratios)) if entropy_ratios else 0.0,
        "mean_confidence": float(np.mean(confidences)) if confidences else 0.0,
    }

fold_logs = []
GENERATION_ERRORS = []

if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
    raise RuntimeError("Model generation is not ready; refusing fallback generation for v24.")

t0 = time.time()
for pi, p in enumerate(PROMPTS, start=1):
    print(f"\n[{pi}/{len(PROMPTS)}] {p[:90]}")
    for role in BRANCH_ROLES:
        bid = f"inverse_{pi:03d}_{role}"
        try:
            fl = generate_fold_log(branch_prompt(p, role), role, bid)
            fold_logs.append(fl)
            print(f"  {role}: tokens={fl['total_tokens']} entropy_mean={fl['mean_entropy_ratio']:.4f} conf_mean={fl['mean_confidence']:.4f} scorer={fl['scorer_term_count']}")
        except Exception as e:
            err = {
                "branch_id": bid,
                "prompt": p,
                "role": role,
                "error": traceback.format_exc()
            }
            GENERATION_ERRORS.append(err)
            print("  ERROR:", bid, repr(e))

elapsed_generation = time.time() - t0
print("\nGenerated branches:", len(fold_logs), "Errors:", len(GENERATION_ERRORS), "Elapsed:", elapsed_generation)



[1/36] design a shape-first retrieval step where no noun match exists but the inverse need is cle
  construct: tokens=120 entropy_mean=0.2126 conf_mean=0.8487 scorer=0
  verify: tokens=120 entropy_mean=0.2047 conf_mean=0.8290 scorer=0

[2/36] how should retrieval work when keywords fail but the operation is obvious
  construct: tokens=120 entropy_mean=0.2027 conf_mean=0.8410 scorer=0
  verify: tokens=120 entropy_mean=0.2023 conf_mean=0.8603 scorer=1

[3/36] explain inverse operational fit for search without noun matching
  construct: tokens=120 entropy_mean=0.2370 conf_mean=0.8066 scorer=0
  verify: tokens=120 entropy_mean=0.2240 conf_mean=0.8104 scorer=0

[4/36] design a verifier for retrieval candidates selected by need rather than label
  construct: tokens=120 entropy_mean=0.1805 conf_mean=0.8847 scorer=0
  verify: tokens=120 entropy_mean=0.1320 conf_mean=0.8766 scorer=0

[5/36] how can an agent rank candidates by function instead of name
  construct: tokens=120 entropy_mean=0.1982

## 4. H-Field Pressure and Nulls

Primary metric:

$$
P_H^{(\text{entropy})}(\sigma)
=
\frac{1}{L}\sum_\ell
\exp\left(-\frac{|H_\ell^{(\text{entropy})}-H|}{\sigma}\right)
$$

Nulls:

1. **Random-target null:** same trajectory, random target values. Tests whether $\pi/9$ is special relative to other targets.
2. **Matched-random null:** synthetic trajectories with same mean/std. Tests whether observed pressure exceeds distributional expectation.
3. **Position-shuffle null:** preserves values, tests whether rolling-window structure is order-specific.


In [5]:

def get_channel_values(fold_log: Dict[str, Any], channel: str) -> np.ndarray:
    if channel == "entropy":
        return np.array([s["entropy_ratio"] for s in fold_log["fold_states"]], dtype=float)
    if channel == "confidence":
        return np.array([s["confidence"] for s in fold_log["fold_states"]], dtype=float)
    raise ValueError(f"Unknown channel: {channel}")

def pressure_summary(values: np.ndarray, sigma: float, target: float = H_TARGET) -> Dict[str, Any]:
    pressure = h_field_pressure(values, target, sigma)
    dists = np.abs(values - target)
    best_idx = int(np.argmin(dists))
    out = {
        "mean_pressure": float(np.mean(pressure)) if len(pressure) else 0.0,
        "median_pressure": float(np.median(pressure)) if len(pressure) else 0.0,
        "max_pressure": float(np.max(pressure)) if len(pressure) else 0.0,
        "best_distance": float(dists[best_idx]) if len(dists) else None,
        "best_position": int(best_idx + 1) if len(dists) else None,
        "best_relative_position": float((best_idx + 1) / len(dists)) if len(dists) else None,
        "best_value": float(values[best_idx]) if len(dists) else None,
    }
    for w in WINDOWS:
        roll = rolling_mean(pressure, w)
        out[f"max_rolling_pressure_w{w}"] = float(np.max(roll)) if len(roll) else 0.0
        out[f"mean_rolling_pressure_w{w}"] = float(np.mean(roll)) if len(roll) else 0.0
    return out

def empirical_p_high(real_value: float, null_values: List[float]) -> float:
    arr = np.array(null_values, dtype=float)
    return float((np.sum(arr >= real_value) + 1) / (len(arr) + 1))

def z_score(real_value: float, null_values: List[float]) -> float:
    arr = np.array(null_values, dtype=float)
    sd = float(np.std(arr))
    if sd < 1e-12:
        return 0.0
    return float((real_value - float(np.mean(arr))) / sd)

def random_target_null(values: np.ndarray, sigma: float, n: int = N_TARGET_NULL) -> Dict[str, Any]:
    # Exclude a narrow region around H so the null target is not accidentally H.
    vals = []
    roll5 = []
    attempts = 0
    while len(vals) < n and attempts < n * 10:
        attempts += 1
        t = random.random()
        if abs(t - H_TARGET) < 0.02:
            continue
        m = pressure_summary(values, sigma, target=t)
        vals.append(m["mean_pressure"])
        roll5.append(m["max_rolling_pressure_w5"])
    return {"mean_pressure": vals, "max_rolling_pressure_w5": roll5}

def matched_random_null(values: np.ndarray, sigma: float, n: int = N_MATCHED_NULL) -> Dict[str, Any]:
    mu = float(np.mean(values))
    sd = float(np.std(values) + 1e-9)
    L = len(values)
    vals = []
    roll5 = []
    for _ in range(n):
        synth = np.random.normal(mu, sd, L)
        synth = np.clip(synth, 0.0, 1.0)
        m = pressure_summary(synth, sigma, target=H_TARGET)
        vals.append(m["mean_pressure"])
        roll5.append(m["max_rolling_pressure_w5"])
    return {"mean_pressure": vals, "max_rolling_pressure_w5": roll5}

def position_shuffle_null(values: np.ndarray, sigma: float, n: int = N_POSITION_NULL) -> Dict[str, Any]:
    # Mean pressure is identical under position shuffle; only rolling window pressure is meaningful.
    roll5 = []
    roll10 = []
    for _ in range(n):
        shuffled = values.copy()
        np.random.shuffle(shuffled)
        m = pressure_summary(shuffled, sigma, target=H_TARGET)
        roll5.append(m["max_rolling_pressure_w5"])
        roll10.append(m["max_rolling_pressure_w10"])
    return {"max_rolling_pressure_w5": roll5, "max_rolling_pressure_w10": roll10}

def analyze_channel(fold_log: Dict[str, Any], channel: str) -> Dict[str, Any]:
    values = get_channel_values(fold_log, channel)
    channel_result = {"channel": channel, "sigmas": {}}

    for sigma in SIGMAS:
        real = pressure_summary(values, sigma, H_TARGET)
        target_null = random_target_null(values, sigma, N_TARGET_NULL)
        matched_null = matched_random_null(values, sigma, N_MATCHED_NULL)
        pos_null = position_shuffle_null(values, sigma, N_POSITION_NULL)

        channel_result["sigmas"][f"sigma_{sigma:g}"] = {
            "sigma": sigma,
            "real": real,
            "random_target_null": {
                "mean_pressure_null_mean": float(np.mean(target_null["mean_pressure"])),
                "mean_pressure_null_std": float(np.std(target_null["mean_pressure"])),
                "mean_pressure_z": z_score(real["mean_pressure"], target_null["mean_pressure"]),
                "mean_pressure_p_high": empirical_p_high(real["mean_pressure"], target_null["mean_pressure"]),
                "max_rolling_w5_null_mean": float(np.mean(target_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_null_std": float(np.std(target_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_z": z_score(real["max_rolling_pressure_w5"], target_null["max_rolling_pressure_w5"]),
                "max_rolling_w5_p_high": empirical_p_high(real["max_rolling_pressure_w5"], target_null["max_rolling_pressure_w5"]),
            },
            "matched_random_null": {
                "mean_pressure_null_mean": float(np.mean(matched_null["mean_pressure"])),
                "mean_pressure_null_std": float(np.std(matched_null["mean_pressure"])),
                "mean_pressure_z": z_score(real["mean_pressure"], matched_null["mean_pressure"]),
                "mean_pressure_p_high": empirical_p_high(real["mean_pressure"], matched_null["mean_pressure"]),
                "max_rolling_w5_null_mean": float(np.mean(matched_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_null_std": float(np.std(matched_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_z": z_score(real["max_rolling_pressure_w5"], matched_null["max_rolling_pressure_w5"]),
                "max_rolling_w5_p_high": empirical_p_high(real["max_rolling_pressure_w5"], matched_null["max_rolling_pressure_w5"]),
            },
            "position_shuffle_null": {
                "max_rolling_w5_null_mean": float(np.mean(pos_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_null_std": float(np.std(pos_null["max_rolling_pressure_w5"])),
                "max_rolling_w5_z": z_score(real["max_rolling_pressure_w5"], pos_null["max_rolling_pressure_w5"]),
                "max_rolling_w5_p_high": empirical_p_high(real["max_rolling_pressure_w5"], pos_null["max_rolling_pressure_w5"]),
                "max_rolling_w10_null_mean": float(np.mean(pos_null["max_rolling_pressure_w10"])),
                "max_rolling_w10_null_std": float(np.std(pos_null["max_rolling_pressure_w10"])),
                "max_rolling_w10_z": z_score(real["max_rolling_pressure_w10"], pos_null["max_rolling_pressure_w10"]),
                "max_rolling_w10_p_high": empirical_p_high(real["max_rolling_pressure_w10"], pos_null["max_rolling_pressure_w10"]),
            }
        }
    return channel_result

def analyze_fold_log(fold_log: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "branch_id": fold_log["branch_id"],
        "branch_role": fold_log["branch_role"],
        "profile": fold_log["profile"],
        "prompt": fold_log["prompt"],
        "total_tokens": fold_log["total_tokens"],
        "scorer_term_count": fold_log["scorer_term_count"],
        "mean_entropy_ratio": fold_log["mean_entropy_ratio"],
        "mean_confidence": fold_log["mean_confidence"],
        "final_text_preview": fold_log["final_text"][:240],
        "channels": {
            "entropy": analyze_channel(fold_log, "entropy"),
            "confidence": analyze_channel(fold_log, "confidence"),
        }
    }

analysis_results = []
t1 = time.time()
for i, fl in enumerate(fold_logs, start=1):
    print(f"[{i}/{len(fold_logs)}] pressure analysis {fl['branch_id']}")
    analysis_results.append(analyze_fold_log(fl))
elapsed_analysis = time.time() - t1
print("Analysis elapsed:", elapsed_analysis)


[1/72] pressure analysis inverse_001_construct
[2/72] pressure analysis inverse_001_verify
[3/72] pressure analysis inverse_002_construct
[4/72] pressure analysis inverse_002_verify
[5/72] pressure analysis inverse_003_construct
[6/72] pressure analysis inverse_003_verify
[7/72] pressure analysis inverse_004_construct
[8/72] pressure analysis inverse_004_verify
[9/72] pressure analysis inverse_005_construct
[10/72] pressure analysis inverse_005_verify
[11/72] pressure analysis inverse_006_construct
[12/72] pressure analysis inverse_006_verify
[13/72] pressure analysis inverse_007_construct
[14/72] pressure analysis inverse_007_verify
[15/72] pressure analysis inverse_008_construct
[16/72] pressure analysis inverse_008_verify
[17/72] pressure analysis inverse_009_construct
[18/72] pressure analysis inverse_009_verify
[19/72] pressure analysis inverse_010_construct
[20/72] pressure analysis inverse_010_verify
[21/72] pressure analysis inverse_011_construct
[22/72] pressure analysis inver

## 5. Compact Summary

Primary readout is:

$$
P_H^{(\text{entropy})}(\sigma=0.01)
$$

with both random-target and matched-random null comparison.


In [6]:

PRIMARY_SIGMA = 0.01
PRIMARY_SIGMA_KEY = f"sigma_{PRIMARY_SIGMA:g}"

summary_rows = []

for ar in analysis_results:
    for channel in ["entropy", "confidence"]:
        sig = ar["channels"][channel]["sigmas"][PRIMARY_SIGMA_KEY]
        real = sig["real"]
        rt = sig["random_target_null"]
        mr = sig["matched_random_null"]
        ps = sig["position_shuffle_null"]

        summary_rows.append({
            "run_id": RUN_ID,
            "branch_id": ar["branch_id"],
            "branch_role": ar["branch_role"],
            "profile": ar["profile"],
            "channel": channel,
            "sigma": PRIMARY_SIGMA,
            "total_tokens": ar["total_tokens"],
            "scorer_term_count": ar["scorer_term_count"],
            "mean_entropy_ratio": ar["mean_entropy_ratio"],
            "mean_confidence": ar["mean_confidence"],

            "mean_pressure": real["mean_pressure"],
            "median_pressure": real["median_pressure"],
            "max_pressure": real["max_pressure"],
            "best_distance": real["best_distance"],
            "best_position": real["best_position"],
            "best_relative_position": real["best_relative_position"],
            "best_value": real["best_value"],
            "max_rolling_pressure_w5": real["max_rolling_pressure_w5"],
            "max_rolling_pressure_w10": real["max_rolling_pressure_w10"],

            "random_target_mean_pressure_z": rt["mean_pressure_z"],
            "random_target_mean_pressure_p_high": rt["mean_pressure_p_high"],
            "random_target_max_rolling_w5_z": rt["max_rolling_w5_z"],
            "random_target_max_rolling_w5_p_high": rt["max_rolling_w5_p_high"],

            "matched_random_mean_pressure_z": mr["mean_pressure_z"],
            "matched_random_mean_pressure_p_high": mr["mean_pressure_p_high"],
            "matched_random_max_rolling_w5_z": mr["max_rolling_w5_z"],
            "matched_random_max_rolling_w5_p_high": mr["max_rolling_w5_p_high"],

            "position_shuffle_max_rolling_w5_z": ps["max_rolling_w5_z"],
            "position_shuffle_max_rolling_w5_p_high": ps["max_rolling_w5_p_high"],

            "final_text_preview": ar["final_text_preview"],
        })

summary_df = pd.DataFrame(summary_rows)

# Aggregate by role/channel.
aggregate = {
    "run_id": RUN_ID,
    "version": "v24",
    "purpose": "inverse_retrieval_h_field_pressure",
    "source": "new_generation",
    "model": MODEL_ID_OR_PATH,
    "model_ready": MODEL_READY,
    "model_generation_ready": MODEL_GENERATION_READY,
    "h_target": H_TARGET,
    "primary_profile": PRIMARY_PROFILE,
    "primary_channel": PRIMARY_CHANNEL,
    "primary_sigma": PRIMARY_SIGMA,
    "prompt_count": len(PROMPTS),
    "branch_count": len(fold_logs),
    "generation_errors": len(GENERATION_ERRORS),
    "total_tokens": int(sum(fl["total_tokens"] for fl in fold_logs)),
    "elapsed_generation_seconds": elapsed_generation,
    "elapsed_analysis_seconds": elapsed_analysis,
    "dependency_status": DEPENDENCY_STATUS,
    "device_info": DEVICE_INFO,
    "summary_by_channel": {},
    "summary_by_role_channel": {},
}

for channel, g in summary_df.groupby("channel"):
    aggregate["summary_by_channel"][channel] = {
        "count": int(len(g)),
        "mean_pressure": float(g["mean_pressure"].mean()),
        "median_pressure": float(g["mean_pressure"].median()),
        "mean_random_target_z": float(g["random_target_mean_pressure_z"].mean()),
        "mean_random_target_p_high": float(g["random_target_mean_pressure_p_high"].mean()),
        "mean_matched_random_z": float(g["matched_random_mean_pressure_z"].mean()),
        "mean_matched_random_p_high": float(g["matched_random_mean_pressure_p_high"].mean()),
        "mean_position_shuffle_w5_z": float(g["position_shuffle_max_rolling_w5_z"].mean()),
        "best_distance_mean": float(g["best_distance"].mean()),
        "best_distance_median": float(g["best_distance"].median()),
    }

for (role, channel), g in summary_df.groupby(["branch_role", "channel"]):
    aggregate["summary_by_role_channel"][f"{role}.{channel}"] = {
        "count": int(len(g)),
        "mean_pressure": float(g["mean_pressure"].mean()),
        "mean_random_target_z": float(g["random_target_mean_pressure_z"].mean()),
        "mean_random_target_p_high": float(g["random_target_mean_pressure_p_high"].mean()),
        "mean_matched_random_z": float(g["matched_random_mean_pressure_z"].mean()),
        "mean_matched_random_p_high": float(g["matched_random_mean_pressure_p_high"].mean()),
        "mean_position_shuffle_w5_z": float(g["position_shuffle_max_rolling_w5_z"].mean()),
        "best_relative_position_mean": float(g["best_relative_position"].mean()),
    }

# Primary decision summary.
primary_df = summary_df[summary_df["channel"] == PRIMARY_CHANNEL].copy()
aggregate["primary_result"] = {
    "channel": PRIMARY_CHANNEL,
    "sigma": PRIMARY_SIGMA,
    "branches": int(len(primary_df)),
    "mean_pressure": float(primary_df["mean_pressure"].mean()) if len(primary_df) else None,
    "mean_random_target_z": float(primary_df["random_target_mean_pressure_z"].mean()) if len(primary_df) else None,
    "mean_random_target_p_high": float(primary_df["random_target_mean_pressure_p_high"].mean()) if len(primary_df) else None,
    "mean_matched_random_z": float(primary_df["matched_random_mean_pressure_z"].mean()) if len(primary_df) else None,
    "mean_matched_random_p_high": float(primary_df["matched_random_mean_pressure_p_high"].mean()) if len(primary_df) else None,
    "mean_position_shuffle_w5_z": float(primary_df["position_shuffle_max_rolling_w5_z"].mean()) if len(primary_df) else None,
    "best_branch": str(primary_df.loc[primary_df["mean_pressure"].idxmax(), "branch_id"]) if len(primary_df) else None,
}

print("Aggregate primary result:")
print(json.dumps(aggregate["primary_result"], indent=2))
display(summary_df.head(20))


Aggregate primary result:
{
  "channel": "entropy",
  "sigma": 0.01,
  "branches": 72,
  "mean_pressure": 0.02332463473342879,
  "mean_random_target_z": 0.27537362511086844,
  "mean_random_target_p_high": 0.20354645354645357,
  "mean_matched_random_z": -0.2857315335948767,
  "mean_matched_random_p_high": 0.5950160950160951,
  "mean_position_shuffle_w5_z": 0.05683950066150053,
  "best_branch": "inverse_006_construct"
}


,run_id,branch_id,branch_role,profile,channel,sigma,total_tokens,scorer_term_count,mean_entropy_ratio,mean_confidence,...,random_target_mean_pressure_p_high,random_target_max_rolling_w5_z,random_target_max_rolling_w5_p_high,matched_random_mean_pressure_z,matched_random_mean_pressure_p_high,matched_random_max_rolling_w5_z,matched_random_max_rolling_w5_p_high,position_shuffle_max_rolling_w5_z,position_shuffle_max_rolling_w5_p_high,final_text_preview
0,rhi_v24_19f4edff63,inverse_001_construct,construct,inverse_retrieval,entropy,0.01,120,0,0.212603,0.848659,...,0.214785,0.507829,0.305694,-0.698936,0.724276,-0.585048,0.729271,-0.596882,0.511489,"To design a ""shape-first"" retrieval step that ..."
1,rhi_v24_19f4edff63,inverse_001_construct,construct,inverse_retrieval,confidence,0.01,120,0,0.212603,0.848659,...,0.643357,-0.837067,0.651349,-0.028999,0.256743,-0.137950,0.284715,-0.268617,1.000000,"To design a ""shape-first"" retrieval step that ..."
2,rhi_v24_19f4edff63,inverse_001_verify,verify,inverse_retrieval,entropy,0.01,120,0,0.204681,0.829033,...,0.028971,1.344935,0.032967,1.695021,0.060939,0.273346,0.321678,-0.958428,0.820180,"To design a ""shape-first"" retrieval system tha..."
3,rhi_v24_19f4edff63,inverse_001_verify,verify,inverse_retrieval,confidence,0.01,120,0,0.204681,0.829033,...,0.509491,0.133707,0.454545,1.898063,0.054945,1.045681,0.172827,-0.414294,0.583417,"To design a ""shape-first"" retrieval system tha..."
4,rhi_v24_19f4edff63,inverse_002_construct,construct,inverse_retrieval,entropy,0.01,120,0,0.202738,0.840981,...,0.058941,1.647984,0.047952,0.260534,0.370629,1.292292,0.112887,1.430313,0.110889,"When keywords fail in a search or query, it's ..."
5,rhi_v24_19f4edff63,inverse_002_construct,construct,inverse_retrieval,confidence,0.01,120,0,0.202738,0.840981,...,0.520480,0.422438,0.336663,3.771459,0.006993,2.481671,0.035964,-0.336690,0.705295,"When keywords fail in a search or query, it's ..."
6,rhi_v24_19f4edff63,inverse_002_verify,verify,inverse_retrieval,entropy,0.01,120,1,0.202268,0.860299,...,0.419580,0.495411,0.250749,-1.456519,0.944056,-0.356300,0.662338,-0.541468,0.530470,When retrieval fails due to keyword issues but...
7,rhi_v24_19f4edff63,inverse_002_verify,verify,inverse_retrieval,confidence,0.01,120,1,0.202268,0.860299,...,0.372627,0.230362,0.404595,2.914232,0.018981,1.560070,0.109890,-0.340484,0.887113,When retrieval fails due to keyword issues but...
8,rhi_v24_19f4edff63,inverse_003_construct,construct,inverse_retrieval,entropy,0.01,120,0,0.237026,0.806611,...,0.034965,2.464609,0.018981,3.066268,0.004995,2.627683,0.014985,0.616114,0.215784,Inverse Operational Fit (IOF) is an algorithm ...
9,rhi_v24_19f4edff63,inverse_003_construct,construct,inverse_retrieval,confidence,0.01,120,0,0.237026,0.806611,...,0.542458,0.465881,0.275724,1.734523,0.068931,1.830223,0.066933,2.542555,0.011988,Inverse Operational Fit (IOF) is an algorithm ...


## 6. Save Exactly Two Files

The bundle contains compact fold logs, all pressure analysis, and aggregate readouts.

The CSV is the branch/channel summary.


In [7]:

bundle = {
    "run_id": RUN_ID,
    "version": "v24",
    "purpose": "inverse_retrieval_h_field_pressure",
    "created_root": str(ROOT),
    "out_dir": str(OUT_DIR),
    "config": {
        "model_id_or_path": MODEL_ID_OR_PATH,
        "run_prompt_limit": RUN_PROMPT_LIMIT,
        "branch_roles": BRANCH_ROLES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "h_target": H_TARGET,
        "primary_profile": PRIMARY_PROFILE,
        "primary_channel": PRIMARY_CHANNEL,
        "sigmas": SIGMAS,
        "primary_sigma": PRIMARY_SIGMA,
        "n_target_null": N_TARGET_NULL,
        "n_matched_null": N_MATCHED_NULL,
        "n_position_null": N_POSITION_NULL,
        "windows": WINDOWS,
    },
    "model_status": {
        "model_ready": MODEL_READY,
        "model_generation_ready": MODEL_GENERATION_READY,
        "model_error": MODEL_ERROR,
        "smoke_text": SMOKE_TEXT,
        "dependency_status": DEPENDENCY_STATUS,
        "device_info": DEVICE_INFO,
    },
    "prompts": PROMPTS,
    "generation_errors": GENERATION_ERRORS,
    "aggregate": aggregate,
    "summary": summary_df.to_dict(orient="records"),
    "analysis_results": analysis_results,
    "fold_logs": fold_logs,
    "interpretation_lock": {
        "h_is_target": False,
        "h_is_readout_destination": True,
        "primary_claim": "continuous H-field pressure in inverse-retrieval entropy channel",
        "crossing": "minimum H-distance",
        "pressure": "mean exp(-abs(x-H)/sigma)",
        "local_pressure": "rolling mean of field pressure",
        "null_separation": "random-target, matched-random, and position-shuffle controls",
    }
}

bundle_out = OUT_DIR / f"{RUN_ID}_bundle.json"
summary_out = OUT_DIR / f"{RUN_ID}_summary.csv"

with open(bundle_out, "w", encoding="utf-8") as f:
    json.dump(bundle, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_out, index=False)

print("Saved exactly two output files:")
print(bundle_out)
print(summary_out)


Saved exactly two output files:
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v24_outputs\rhi_v24_19f4edff63_bundle.json
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v24_outputs\rhi_v24_19f4edff63_summary.csv


## 7. Interpretation Guide

### Strong positive

$$
z_{\text{random-target}}>2
\quad\text{and}\quad
z_{\text{matched-random}}>2
$$

for the pre-registered entropy channel.

### Distribution-only signal

If random-target is strong but matched-random is weak, H may be special relative to other target values but not stronger than a trajectory with the same mean/std.

### Order/locality signal

If position-shuffle rolling pressure is strong, the H-neighborhood is locally clustered in the real token order.

### Null-like

If all z-scores are near 0 or negative:

$$
\boxed{
\text{H-crossing/pressure is not distinguishable from null under this observable.}
}
$$

That is still useful: it closes a false path and forces the next observable.
